In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

In [3]:
from src.models.cbam_cnn import (
    CBAMCNN
)

In [4]:
from src.training.trainer import (
    train_one_epoch,
    validate
)

from src.preprocessing.dataset_loader import (
    build_dataset_index
)

from src.preprocessing.splitter import (
    create_stratified_split
)

from src.preprocessing.dataloaders import (
    create_datasets,
    create_dataloaders
)

In [5]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [6]:
train_paths, train_labels = build_dataset_index(
    "../data/raw/train"
)

test_paths, test_labels = build_dataset_index(
    "../data/raw/test"
)

In [7]:
(
    X_train,
    X_val,
    y_train,
    y_val
) = create_stratified_split(
    train_paths,
    train_labels
)

In [8]:
(
    train_dataset,
    val_dataset,
    test_dataset
) = create_datasets(
    X_train,
    y_train,
    X_val,
    y_val,
    test_paths,
    test_labels
)

In [9]:
(
    train_loader,
    val_loader,
    test_loader
) = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32
)

In [10]:
counter = Counter(y_train)

print(counter)

Counter({1: 3099, 0: 1073})


In [11]:
total = sum(counter.values())

w_normal = total / (
    2 * counter[0]
)

w_pneumonia = total / (
    2 * counter[1]
)

In [12]:
from collections import Counter
import numpy as np

class_counts = Counter(train_labels)
total_samples = sum(class_counts.values())
class_weights = {
    cls: total_samples / (len(class_counts) * count) 
    for cls, count in class_counts.items()
}

weight_tensor = torch.FloatTensor([class_weights[0], class_weights[1]]).to(device)

In [13]:
model = CBAMCNN()

model = model.to(device)

In [14]:
criterion = nn.CrossEntropyLoss(weight=weight_tensor)

In [15]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='max', 
    factor=0.5, 
    patience=3,
    verbose=True
)

c:\Users\gagan\Desktop\Federated Learning\federated-xray-classification\venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [16]:
NUM_EPOCHS = 20

best_val_acc = 0.0

In [17]:
history = {

    "train_loss": [],
    "train_acc": [],

    "val_loss": [],
    "val_acc": []
}

In [18]:
for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    
    scheduler.step(val_acc)

    history["train_loss"].append(
        train_loss
    )

    history["train_acc"].append(
        train_acc
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_acc"].append(
        val_acc
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "../src/models/best_cbam_cnn.pth"
        )

        print(
            f"Best model saved "
            f"(Val Acc={val_acc:.4f})"
        )

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )

0.0003738845407497138 0.9988345503807068
0.00232497020624578 0.880588710308075
1.762674583005719e-05 0.9997568726539612
0.0011572702787816525 0.886651873588562
2.1020496205892414e-06 0.9999667406082153
0.0013773515820503235 0.9109736680984497
2.429285814287141e-06 0.9999711513519287
0.001174077158793807 0.965021014213562
5.687905286322348e-06 0.9999792575836182
0.0004793874395545572 0.9544584155082703
2.6775528567668516e-06 0.9999603033065796
0.0010230951011180878 0.9694878458976746
1.1571450464487043e-08 0.9999996423721313
4.645702938432805e-05 0.993289053440094
4.3904631041868925e-08 0.9999990463256836
0.0002758931659627706 0.9880380630493164
6.051128366379999e-06 0.9999699592590332
0.0020900964736938477 0.9866551160812378
2.023614342760993e-06 0.9999914169311523
0.0023395237512886524 0.9669683575630188
2.3397712993755704e-06 0.9999915361404419
0.0006487580249086022 0.9851464629173279
1.0431951523059979e-05 0.9999657869338989
0.0037124804221093655 0.9913698434829712
0.000515636405907

In [19]:
import json

with open(
    "../results/metrics/cbam_history.json",
    "w"
) as f:

    json.dump(
        history,
        f,
        indent=4
    )